# Apple diluted EPS, last 12 quarters

> **From:** Managing Director  **To:** You  **Sent:** Tue 21:47
>
> "Need Apple's diluted EPS for the last 12 quarters, in a chart and an Excel file, on my desk by 8am."

**This notebook was written by AI (Claude) from this prompt:**

*"Write a Jupyter notebook for Google Colab that downloads Apple's diluted earnings per share (EPS) from the SEC website (XBRL company facts API), for the last 12 fiscal quarters. Apple does not report a fourth quarter on its own, so calculate it as the full year minus the first three quarters and mark those quarters clearly as derived. Show the numbers in a table, draw a bar chart that is easy to read on a projector, and save an Excel file with a sheet for the data and a sheet with notes on the source. Keep each step short and use simple pandas code with comments in plain English. Add a checks section that compares the derived numbers with what Apple actually reported."*

In [ ]:
# Step 1 - Tools we need (pandas = tables, matplotlib = charts, requests = web download)
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import date

# The SEC asks every user to put a contact in the request header (this one is a placeholder)
HEADERS = {"User-Agent": "GEM course demo demo@example.com"}
URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json"  # CIK 0000320193 = Apple

In [ ]:
# Step 2 - Download everything Apple has filed with the SEC, as data
response = requests.get(URL, headers=HEADERS)
response.raise_for_status()
facts = response.json()

# Keep only diluted EPS (one row per reported period)
eps = pd.DataFrame(facts["facts"]["us-gaap"]["EarningsPerShareDiluted"]["units"]["USD/shares"])
eps["start"] = pd.to_datetime(eps["start"])
eps["end"] = pd.to_datetime(eps["end"])
eps["days"] = (eps["end"] - eps["start"]).dt.days

# The same number appears in several filings (old numbers are repeated as comparisons).
# If two copies differ, we keep the one from the most recent filing.
eps = eps.sort_values("filed").drop_duplicates(["start", "end"], keep="last")

# Keep three-month periods (about 91 days) and full-year periods (about 365 days)
quarters = eps[(eps["days"] >= 80) & (eps["days"] <= 100)].copy()
years = eps[eps["days"] >= 350].copy()

In [ ]:
# Step 3 - Apple's financial year ends in late September: Oct-Dec is fiscal Q1
quarters["fy"] = quarters["end"].dt.year
quarters.loc[quarters["end"].dt.month >= 10, "fy"] += 1  # Oct-Dec belongs to next fiscal year
# Quarters end in late Dec/Mar/Jun, or a few days later (Jan/Apr/Jul) in some years
quarters["q"] = quarters["end"].dt.month.map({12: 1, 1: 1, 3: 2, 4: 2, 6: 3, 7: 3})
quarters["source"] = "10-Q"
years["fy"] = years["end"].dt.year

# Apple reports no fourth quarter. Derived Q4 = full year minus (Q1 + Q2 + Q3).
q4_rows = []
for _, year in years.iterrows():
    first_three = quarters[quarters["fy"] == year["fy"]]
    if len(first_three) == 3:
        q4_value = year["val"] - first_three["val"].sum()
        q4_rows.append({"fy": year["fy"], "q": 4, "end": year["end"], "val": q4_value,
                        "source": "derived: year minus Q1-Q3"})

all_q = pd.concat([quarters, pd.DataFrame(q4_rows)]).sort_values("end").tail(12)
table = pd.DataFrame({
    "Fiscal quarter": "FY" + (all_q["fy"] % 100).astype(str) + " Q" + all_q["q"].astype(int).astype(str),
    "Period end": all_q["end"].dt.date,
    "Diluted EPS (USD)": all_q["val"].round(2),
    "Source": all_q["source"],
}).reset_index(drop=True)
table

In [ ]:
# Step 4 - Bar chart (solid bars = reported in a 10-Q, light hatched bars = derived Q4)
plt.rcParams.update({"font.size": 15})
fig, ax = plt.subplots(figsize=(14, 7))
derived = table["Source"] != "10-Q"
colors = ["#C9D3F0" if d else "#1F3A93" for d in derived]
bars = ax.bar(table["Fiscal quarter"], table["Diluted EPS (USD)"], color=colors,
              edgecolor="#1F3A93", linewidth=1.5)
for bar, d in zip(bars, derived):
    if d:
        bar.set_hatch("//")

ax.bar_label(bars, fmt="%.2f", padding=3)  # write the value above each bar
ax.set_title("Apple diluted EPS, last 12 fiscal quarters", fontsize=20, pad=15)
ax.set_ylabel("Diluted EPS (USD)")
ax.legend(handles=[mpatches.Patch(color="#1F3A93", label="Reported (10-Q)"),
                   mpatches.Patch(facecolor="#C9D3F0", edgecolor="#1F3A93", hatch="//",
                                  label="Derived Q4 (year minus Q1-Q3)")], loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("apple_diluted_eps.png", dpi=200)  # also save the picture as a file
plt.show()

In [ ]:
# Step 5 - Excel file with two sheets: the data and the notes
notes = pd.DataFrame({"Note": [
    "Source: SEC XBRL company facts API, fact EarningsPerShareDiluted",
    "URL: " + URL,
    "Date pulled: " + str(date.today()),
    "Q4 rows marked 'derived' = full-year EPS minus Q1, Q2 and Q3 EPS.",
    "Why: Apple reports no stand-alone Q4. Share count changes each quarter, so derived Q4 is only an approximation.",
]})
with pd.ExcelWriter("apple_diluted_eps.xlsx") as writer:
    table.to_excel(writer, sheet_name="EPS", index=False)
    notes.to_excel(writer, sheet_name="Notes", index=False)
    for column, width in {"A": 16, "B": 14, "C": 20, "D": 28}.items():
        writer.sheets["EPS"].column_dimensions[column].width = width
    writer.sheets["Notes"].column_dimensions["A"].width = 110

# In Google Colab this downloads the files to your computer; on a laptop it is skipped
try:
    from google.colab import files
    files.download("apple_diluted_eps.xlsx")
    files.download("apple_diluted_eps.png")
except ImportError:
    print("Not in Colab - files are saved next to the notebook.")

In [ ]:
# Step 6 - Checks: would you send this to your MD?
assert len(table) == 12, "We expected 12 quarters"

# Check A: for each full year, Q1+Q2+Q3+Q4 must equal the reported full-year EPS
for _, year in years.iterrows():
    in_year = all_q[all_q["fy"] == year["fy"]]
    if len(in_year) == 4:
        print("FY", year["fy"], "sum of 4 quarters:", round(in_year["val"].sum(), 2),
              "| reported full year:", year["val"])
        assert abs(in_year["val"].sum() - year["val"]) < 0.005

# Check B (the real test): derived Q4 versus Q4 EPS in Apple's own earnings press release.
# Sources: Apple 8-K, Exhibit 99.1, on sec.gov, filed 2025-10-30 (FY25 and FY24 Q4):
#   https://www.sec.gov/Archives/edgar/data/320193/000032019325000077/a8-kex991q4202509272025.htm
# and filed 2026-11-02 (FY26 Q4):
#   https://www.sec.gov/Archives/edgar/data/320193/000032019326000024/a8-kex991q4202609262026.htm
apple_reported_q4 = {"FY24 Q4": 0.97, "FY25 Q4": 1.85, "FY26 Q4": 2.10}
for label, reported in apple_reported_q4.items():
    row = table[table["Fiscal quarter"] == label]
    if len(row) == 0:
        continue  # this quarter is no longer in the last 12
    mine = row["Diluted EPS (USD)"].iloc[0]
    print(label, "| derived:", mine, "| Apple reported:", reported, "| gap:", round(mine - reported, 2))

**Would you send this to your MD?**

- Q1-Q3 numbers are exactly what Apple reported in its 10-Qs.
- Q4 is our calculation. It can be off by a cent (see the check above: FY25 Q4 is 1.84 derived vs 1.85 reported, FY26 Q4 is 2.11 vs 2.10) because the share count changes every quarter.
- FY24 Q4 (0.97) is low because of a one-time tax charge of USD 10.2 billion, not because the business weakened. A good analyst says this in the email.
- These are GAAP numbers, not "adjusted" numbers. Check what your MD wants.